# Mutinomial Logistic Regression
**Multinomial Logistic Regression** solves the multiclass problem with:
 - **K classes**: labels $C_1, C_2, \dots, C_k$
 -  $n$ **observations**: $(x^{(1)}, y^{(1)}), (x^{(2)}, y^{(2)}), \dots (x^{(n)}, y^{(n)})$
 -  **Feature vector**: $x^{(i)} = [x^{(i)}_0, x^{(i)}_1, \dots, x^{(i)}_{d+1}] \in \mathbb R^{d+1}$ ($(d +1)$-dimension feature vector with $x^{(i)}_0 = 1$) is a feature vector for observation $i$.
 -  **Class labels**: $y^{(i)}= [y^{(i)}_1, y^{(i)}_2, \dots, y^{(i)}_K]$ represents class labels as *one-hot encoded* vector, where $y^{(i)}_k = 1$ if observation $i$ belongs to class $C_k$, and $y^{(i)}_k = 0$ otherwise.

We, begin with a linear predictor for each observation and class $C_k$.
$$
\eqalign {
z_{ik} &= {\theta^{(k)}}^T \cdot x^{(i)}\\
&= \theta^{(k)}_0 \cdot x^{(i)}_0 + \theta^{(k)}_1 \cdot x^{(i)}_1 + \cdots + \theta^{(k)}_d \cdot x^{(i)}_d
}
$$
where, 
- $\theta^{(k)}$ is weights for class $k$ ($\theta^{(k)}_0$ is the bias)
- $x^{(i)}$ represents the feature vector for observation $i$
- $z_{ik}$ represents the *"row score"* or the *"logits"* for observation $i$ for class $k$

## Hypothesis Function (Softmax)

Using the Baye's rule to caculate posterior probability
$$
\eqalign {
\mathbb P(y^{(i)} = C_k | x^{(i)}; \theta) &= \frac {\mathbb P (x^{(i)}| y^{(i)} = C_k)\mathbb P(y^{(i)})}{\sum_{j=1}^K \mathbb P(x^{(i)}|y^{(i)} = C_j)\mathbb P(y^{(i)} = C_j)}\\
p_{ik} &= \frac {\exp(z_{ik})}{\sum_{j=1}^K \exp(z_{ij})}
}
$$

where,
- $z_{ik} = \ln\mathbb P (x^{(i)}| y^{(i)} = C_k)\mathbb P(y^{(i)}) = \theta_k^T\cdot x^{(i)}$.
- $p_{ik}$ is probability that observation $i$ belongs to class $C_k$, given $x^{(i)}$ and $\theta$.

```python
def softmax(z: np.array):
    proba = np.zeros_like(z, dtype=float)
    x_exp = np.exp(z)
    exp_sum = np.sum(z_exp)
    dim = z.shape[0]
    for i in range(dim):
        proba[i] = z_exp[i] / exp_sum
    return proba
```

# Cost Function - Cross Entrophy
Using $MLE$ to pick the best $\theta$ using the likelihood function. For a single observation $i$ with the true class label represented by the *one-hot vector* $y_i$, the probability of observing this outcome under out model is:
$$
\mathbb P(y_i|x_i; \theta) = \prod_{k=1}^K (p_{ik})^{y^{(i)}_k}
$$
This is a multinomial probability. Since exactly one $y_{ik} = 1$ and all others are $0$, this product equal $p_{ik}$ for the true class $C_k$ and $1$ for every other classes.
Now for the whole dataset we will use the likelihood function:
$$
\eqalign {
L(\theta) &= \mathbb P(y|x; \theta)\\
&= \prod_{i=1}^n \mathbb P(y^{(i)}|x^{(i)};\theta)\\
&= \prod_{i=1}^n \prod_{k=1}^K (p_{ik})^{y^{(i)}_{k}}\\
L(\theta) &= \prod_{i=1}^n \prod_{k=1}^K \left(\frac {\exp(z_{ik})}{\sum_{j=1}^K \exp(z_{ij})}\right)^{y_ik}
}
$$
Taking the logarithm of the likelihood function:
$$
\small
\eqalign {
\ell(\theta) &= \ln L(\theta) = \ln \left(\prod_{i=1}^n \prod_{k=1}^K p_{ik}^{y^{(i)}_{k}}\right)\\
&= \sum_{i=1}^n \sum_{k=1}^K \ln \left((p_{ik})^{y^{(i)}_{k}}\right)\\ &= \sum_{i=1}^n \sum_{k=1}^K y_{ik} \ln \left(p_{ik}\right)\\
&=\sum_{i=1}^n \ln \left(p_{iy_i}\right)\\
&= \sum_{i=1}^n \ln \left(\frac {\exp(z_{iy_i})}{\sum_{j=1}^K \exp(z_{ij})} \right)\\
&= \sum_{i=1}^n \left[\ln(\exp(z_{iy_i})) - \ln\left({\textstyle\sum_{j=1}^K \exp(z_{ij}})\right) \right]\\
&= \sum_{i=1}^n \left[z_{iy_i} - \ln\left({\textstyle\sum_{j=1}^K \exp(z_{ij}})\right) \right]\\
}
$$
where,
- $y_i$ is the correct class ($C_k$) for the observation $i$.
- $p_{iy_i}$ is the probability that observation  belongs correct class $y_i$. 
- $z_{iy_i}$ is the linear predictor for the correct class $y_i$ for observation $i$.

Our final log-likelihood function looks like:
$$
\ell(\theta) = {\textstyle\sum_{i=1}^n \left[z_{iy_i} -  \ln\left({\sum_{j=1}^K \exp(z_{ij})}\right)  \right]}
$$
We want to **maximize** $\ell(\theta)$ which is same as **minimizing** $-\ell(\theta)$.
$$
\eqalign {
{\underset \theta \max}~~ \ell(\theta) &= {\underset \theta \min} ~ -\ell(\theta)\\ 
&= -  {\textstyle\sum_{i=1}^n \left[z_{iy_i} -  \ln\left({\sum_{j=1}^K \exp(z_{ij})}\right)  \right]}\\
{\underset \theta \min}~ J(\theta) = {\underset \theta \min }~-\ell(\theta) &= {\textstyle\sum_{i=1}^n \left[  \ln\left({\sum_{j=1}^K \exp(z_{ij})}\right) - z_{iy_i} \right]}
}
$$
To make the loss independent of dataset size, we typically use the average loss:
$$
\eqalign {
\text{Cost}(\theta) &= {\textstyle \frac 1n\sum_{i=1}^n \left[  \ln\left({\sum_{j=1}^K \exp(z_{ij})}\right) - z_{ik} \right]}\\
&= {\textstyle \frac 1n \sum_{i=1}^n \left[\sum_{j=1}^K \ln (\exp(z_{ij})) - z_{ik}\right]}\\
&= {\textstyle \frac 1n \sum_{i=1}^n \left[\sum_{j=1}^k z_{ij} - z_{ik}\right]}
}
$$
where, 
- $z_{ik}$ is the raw score for correct class $k$ for observation $i$

In [2]:
import numpy as np

In [3]:
def softmax(z: np.array):
    proba = np.zeros_like(z, dtype=float)
    z_exp = np.exp(z)
    exp_sum = np.sum(z_exp)
    dim = z.shape[0]
    for i in range(dim):
        proba[i] = z_exp[i] / exp_sum
    return proba

def hFunc(Theta, X):
    Z = np.dot(X, Theta.T)
    return Z, np.apply_along_axis(softmax, 1, Z)

In [51]:
class MultinomialLogisticRegression:
    def init(self):
        self.Theta = None
        self.cost_history = []

    def _shuffle_data(self, X, Y):
        """
        Shuffles the data.
        """
        p = np.random.permutation(len(X))
        return X[p], Y[p]

    def _softmax(self, z: np.array):
        proba = np.zeros_like(z, dtype=float)
        z_exp = np.exp(z)
        exp_sum = np.sum(z_exp)
        dim = z.shape[0]
        for i in range(dim):
            proba[i] = z_exp[i] / exp_sum
        return proba
    
    def _hFunc(self, X):
        Z = np.dot(X, self.Theta.T)     # shape = (n_rows, n_class)
        return Z, np.apply_along_axis(self._softmax, 1, Z)
        
    def _mlr_calc(self, X, Y):
        Z, probas = self._hFunc(X)
        n_rows, n_class = Y.shape
        total_cost = 0.0
        for i in range(n_rows):
            z = Z[i]
            z_k = np.dot(z, Y[i].T)    # Raw score for correct class
            z_sum = np.sum(z)
            total_cost += z_sum - z_k
        cost = total_cost / n_rows
        
        sub = probas - Y    # (p_{ik} - y_{ik})
        grad = np.dot(X.T, sub).T
        avg_grad = grad / n_rows

        return cost, avg_grad
    
    def fit(self, x, y, epoch=5, batch_size=5):
        """
        Args:
            X: 2D data array
            Y: 2D one-hot encoded array of labels
            epoch: number of epoch
            batch_size: Batch size for training
        """
        
        X = np.array(x)
        Y = np.array(y)
        
        n_rows, n_feat = X.shape
        Xb = np.concatenate((np.ones(shape=(n_rows, 1)), X), axis=1)
        _, n_class = Y.shape
        self.Theta = np.zeros((n_class, n_feat + 1), dtype=float)
        n = lambda i: (1 + i) ** -1
        self.cost_history = []
        
        for i in range(epoch):
            j = 1
            sX, sY = self._shuffle_data(Xb, Y)
            for start in range(0, n_rows, batch_size):
                # Calculate and Get Gradient and cost
                cost, avg_grad = self._mlr_calc(Xb, Y)
                
                # Update weights
                self.Theta -= n(i) * avg_grad
                
                # Update cost history
                self.cost_history.append(cost)
                
                j += 1

        return self

In [5]:
X = np.array([[1, 1, 0], [0, 1, 1], [1, 0, 1]])
Y = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
Theta = np.array([[0.5, 0.5, 0], [0, 0.5, 0.5], [0.5, 0, 0.5]])

In [6]:
Z, probas = hFunc(Theta, X)
Z, probas

(array([[1. , 0.5, 0.5],
        [0.5, 1. , 0.5],
        [0.5, 0.5, 1. ]]),
 array([[0.45186276, 0.27406862, 0.27406862],
        [0.27406862, 0.45186276, 0.27406862],
        [0.27406862, 0.27406862, 0.45186276]]))

In [7]:
total_cost = 0.0
for i in range(3):
    z = Z[i]
    z_k = np.dot(z, Y[i].T)    # Raw score for correct class
    z_sum = np.sum(z)
    print(z_sum - z_k, end="  ")
    total_cost += z_sum - z_k
cost = total_cost / 3
cost

1.0  1.0  1.0  

np.float64(1.0)

In [8]:
sub = probas - Y
sub

array([[-0.54813724,  0.27406862,  0.27406862],
       [ 0.27406862, -0.54813724,  0.27406862],
       [ 0.27406862,  0.27406862, -0.54813724]])

In [9]:
X.T

array([[1, 0, 1],
       [1, 1, 0],
       [0, 1, 1]])

In [14]:
np.dot(X.T, sub).T

array([[-0.27406862, -0.27406862,  0.54813724],
       [ 0.54813724, -0.27406862, -0.27406862],
       [-0.27406862,  0.54813724, -0.27406862]])

In [21]:
newX = np.concatenate((np.ones(shape=(3,1)), X), axis=1, dtype=float)

In [22]:
newX

array([[1., 1., 1., 0.],
       [1., 0., 1., 1.],
       [1., 1., 0., 1.]])

In [52]:
model = MultinomialLogisticRegression()

In [80]:
model.fit(X, Y, epoch=100, batch_size=1)

In [81]:
model.Theta

array([[ 2.88033468e-17,  8.25828164e-01,  8.25828164e-01,
        -1.65165633e+00],
       [ 2.88033468e-17, -1.65165633e+00,  8.25828164e-01,
         8.25828164e-01],
       [ 1.92982816e-17,  8.25828164e-01, -1.65165633e+00,
         8.25828164e-01]])

In [82]:
model.cost_history

[np.float64(0.0),
 np.float64(-0.22222222222222202),
 np.float64(-0.41855401254713365),
 np.float64(-0.5906617584348046),
 np.float64(-0.6659846742144169),
 np.float64(-0.7366735000251955),
 np.float64(-0.8030844127398821),
 np.float64(-0.844734054371894),
 np.float64(-0.8847701637391167),
 np.float64(-0.92328152791241),
 np.float64(-0.9510847201877809),
 np.float64(-0.9781206828869665),
 np.float64(-1.004421278848299),
 np.float64(-1.0248978399650674),
 np.float64(-1.0449412427591518),
 np.float64(-1.0645657551307404),
 np.float64(-1.080581890806135),
 np.float64(-1.0963255773275185),
 np.float64(-1.111804198369607),
 np.float64(-1.1248505177396662),
 np.float64(-1.137712297470178),
 np.float64(-1.1503937659259773),
 np.float64(-1.1613358768082458),
 np.float64(-1.1721460734242082),
 np.float64(-1.1828269649256002),
 np.float64(-1.1922084161042656),
 np.float64(-1.2014916324079699),
 np.float64(-1.2106783186900165),
 np.float64(-1.2188609607254786),
 np.float64(-1.2269680605835143),
 

In [83]:
model._hFunc(newX)

(array([[ 1.65165633, -0.82582816, -0.82582816],
        [-0.82582816,  1.65165633, -0.82582816],
        [-0.82582816, -0.82582816,  1.65165633]]),
 array([[0.85623161, 0.07188419, 0.07188419],
        [0.07188419, 0.85623161, 0.07188419],
        [0.07188419, 0.07188419, 0.85623161]]))